# Focus Group Transcripts and Social Cognitive Theory Coding for Adolescent Social Media Engagement Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.cntx-bwec/fair2.json
```

In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.
We start by importing the required packages and loading the dataset's metadata.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cntx-bwec/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata as an object
metadata = dataset.metadata

# Print basic dataset information
print(f"{metadata.name}: {metadata.description}")

# Optionally, for further inspection, print keys
print("\nMetadata fields:")
for field in dir(metadata):
    if not field.startswith('_'):
        print(field)

## 2. Data Overview
Review available record sets and their fields using the `@id` of each entity.

A Croissant dataset organizes tabular data into record sets (tables), each with fields (columns), each referenced via their unique `@id`.

In [ ]:
# List all record sets and their fields by @id
record_sets = dataset.record_sets

print(f"Number of record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")
    print("  Fields:")
    for fld in rs.get('fields', []):
        print(f"    Field @id: {fld['@id']}, name: {fld.get('name', 'N/A')}, dataType: {fld.get('dataType', 'N/A')}")
    print()

# Optionally, print the first few records from each record set
for rs in record_sets:
    rs_id = rs['@id']
    print(f"Records from RecordSet {rs_id}:")
    for x in dataset.records(record_set=rs_id):
        print(x)
        break  # print only the first record for brevity

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. Use record set and field `@id`s identified above.

For demonstration, all record sets will be loaded, but you can focus on specific sets or fields as needed.

In [ ]:
# Extract data from each record set using their @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nRecordSet {record_set_id}: columns:")
    print(df.columns.tolist())
    print(df.head())

# Select a primary record set for further EDA -- here, the first record set
main_record_set_id = record_set_ids[0]
df_main = dataframes[main_record_set_id]

print(f"\nPrimary DataFrame columns for {main_record_set_id}:")
print(df_main.columns.tolist())
df_main.head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, and grouping based on record set and field `@id`s.

Here we demonstrate numeric field filtering and normalization; you may adapt to the actual fields present in your dataset.

In [ ]:
# Inspect columns and select a numeric field by @id for demonstration
print("Fields in DataFrame:", df_main.columns.tolist())

# Attempt to identify a numeric field -- replace with actual @id if known
numeric_field_id = None
for col in df_main.columns:
    if pd.api.types.is_numeric_dtype(df_main[col]):
        numeric_field_id = col
        print(f"Using numeric field: {numeric_field_id}")
        break

if numeric_field_id is not None:
    threshold = 10
    filtered_df = df_main[df_main[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Choose a group field by @id (categorical) for demonstration
    group_field_id = None
    for col in df_main.columns:
        if (df_main[col].dtype == 'object' or pd.api.types.is_categorical_dtype(df_main[col])) and col != numeric_field_id:
            group_field_id = col
            print(f"Grouping by field: {group_field_id}")
            break

    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA. Please check dataset columns.")

## 5. Visualization
Visualize distributions or relationships between fields.

Here we plot a histogram for a numeric field and, if available, a bar chart for group-wise means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df_main[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id is not None and group_field_id in grouped_df.columns:
        plt.figure(figsize=(10,4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
        plt.title(f"Average {numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean of {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.show()

## 6. Conclusion
In this notebook, we've successfully loaded the focus group transcript dataset with Social Cognitive Theory coding using `mlcroissant`. We explored metadata, inspected record sets and fields via their `@id`, extracted and analyzed tabular contents, and visualized data distributions.

**Key findings:**
- The dataset is structured with multiple record sets and annotated fields, accessible by their `@id`.
- Data processing scripts can flexibly filter, normalize, and group information using Croissant metadata.
- Thematic coding and participant quotations are machine-readable and ready for analytical workflows.

This dataset enables qualitative research into adolescent social media engagement and can support a variety of reproducible analyses. Continue exploring deeper relationships or apply custom preprocessing as needed!